# 正常 / 幻觉图样本：graph-level PCA + t-SNE

每个点是一张图。标签只用于上色，不参与 PCA / t-SNE。
先把变长图变成固定长度结构向量，再降维；同时按 response length 着色检查长度混杂。

In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

TRACE_SPLIT = Path("/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/model_traces/llama31_8b/test")
GRAPH_ROOT = Path("/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/graphs/llama31_8b/relation_topk_channels/test")

MAX_PER_CLASS = 500
RANDOM_STATE = 42
OUTPUT_DIR = Path("viz_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
def jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def label_map(path):
    out = {}
    for r in jsonl(path):
        sid = str(r["sample_id"])
        if "positive_runs" in r:
            out[sid] = int(bool(r["positive_runs"]))
        elif "response_label" in r:
            out[sid] = int(r["response_label"])
        else:
            out[sid] = int(r["label"])
    return out

def load_graph(path):
    g = torch.load(path, map_location="cpu", weights_only=True)
    return g["graph"] if isinstance(g, dict) and "graph" in g else g

index = jsonl(GRAPH_ROOT / "index.jsonl")
labels = label_map(TRACE_SPLIT / "labels.jsonl")
rows = [r for r in index if str(r["sample_id"]) in labels]

rng = np.random.default_rng(RANDOM_STATE)
by_y = {y:[r for r in rows if labels[str(r["sample_id"])] == y] for y in (0,1)}
n = min(MAX_PER_CLASS, len(by_y[0]), len(by_y[1]))
selected = []
for y in (0,1):
    ids = rng.choice(len(by_y[y]), n, replace=False)
    selected += [by_y[y][i] for i in ids]
rng.shuffle(selected)

print("selected:", len(selected), "correct:", n, "hallucinated:", n)

## 固定长度 graph-level 结构向量

默认特征：图规模、PR/RR密度、source复用/集中度、response indegree、RR局部性、边强度、每边活跃attention channel数。
这些量不需要训练，适合先检验“拓扑本身是否含 correctness signal”。

In [ ]:
def mean(x): return float(x.mean()) if len(x) else 0.0
def std(x): return float(x.std(unbiased=False)) if len(x) else 0.0
def q(x,p): return float(torch.quantile(x.float(),p)) if len(x) else 0.0

def edge_strength(g):
    E = g["edge_index"].shape[1]
    if E == 0: return torch.empty(0)
    if "edge_weight" in g: return g["edge_weight"].float()
    if "edge_ptr" in g and "edge_value" in g:
        ptr, val = g["edge_ptr"].long(), g["edge_value"].float()
        return torch.tensor([val[ptr[e]:ptr[e+1]].max() if ptr[e+1]>ptr[e] else 0.0 for e in range(E)])
    return torch.ones(E)

FEATURE_NAMES = [
    "num_nodes","response_ratio","edges_per_response",
    "pr_ratio","rr_ratio","pr_density","rr_density",
    "unique_source_ratio","max_source_share","source_hhi",
    "indegree_mean","indegree_std",
    "rr_lag_mean","rr_local4","rr_local8",
    "edge_strength_mean","edge_strength_std","edge_strength_p90",
    "channels_per_edge_mean","channels_per_edge_std",
]

def graph_vector(g):
    if "edge_index" not in g:
        raise ValueError("这个 notebook 当前用于 token graph；请先选择 original/relation_topk(_channels) 图目录。")
    x, ei, et = g["x"].float(), g["edge_index"].long(), g["edge_type"].long()
    N, p = x.shape[0], int(g["response_idx"])
    R, E = N-p, ei.shape[1]
    src, tgt = ei
    pr, rr = et==0, et==1
    npr, nrr = int(pr.sum()), int(rr.sum())

    if E:
        _, c = torch.unique(src, return_counts=True)
        prob = c.float()/c.sum()
        hhi = float((prob*prob).sum())
        hhi = 1.0 if len(c)==1 else (hhi-1/len(c))/(1-1/len(c))
        unique_ratio = len(c)/max(1,N)
        max_share = float(c.max()/c.sum())
        indeg = torch.bincount((tgt-p).clamp_min(0), minlength=max(1,R)).float()[:R]
    else:
        unique_ratio=max_share=hhi=0.0
        indeg=torch.zeros(R)

    if nrr:
        lag=(tgt[rr]-src[rr]).float()
        lag_mean=float((lag/max(1,R)).mean())
        local4=float((lag<=4).float().mean())
        local8=float((lag<=8).float().mean())
    else:
        lag_mean=local4=local8=0.0

    s=edge_strength(g)
    if "edge_ptr" in g:
        ptr=g["edge_ptr"].long()
        channels=(ptr[1:]-ptr[:-1]).float()
    else:
        channels=torch.ones(E)

    v=[
        N, R/max(1,N), E/max(1,R),
        npr/max(1,E), nrr/max(1,E),
        npr/max(1,p*R), nrr/max(1,R*max(0,R-1)//2),
        unique_ratio,max_share,hhi,
        mean(indeg),std(indeg),
        lag_mean,local4,local8,
        mean(s),std(s),q(s,.9),
        mean(channels),std(channels),
    ]
    return np.asarray(v,dtype=np.float32)

In [ ]:
X, rec = [], []
for r in selected:
    sid=str(r["sample_id"])
    g=load_graph(GRAPH_ROOT/r["path"])
    X.append(graph_vector(g))
    N=g["x"].shape[0]; p=int(g["response_idx"])
    rec.append({
        "sample_id":sid,"source_id":str(r.get("source_id","")),
        "label":labels[sid],"response_length":N-p,
        "graph_path":str(GRAPH_ROOT/r["path"]),
    })

X=np.stack(X)
meta=pd.DataFrame(rec)
y=meta.label.to_numpy()

Xs=StandardScaler().fit_transform(X)
d=min(50,Xs.shape[1],Xs.shape[0]-1)
Xp=PCA(n_components=d,random_state=RANDOM_STATE).fit_transform(Xs)
perplexity=min(30,max(5,(len(Xp)-1)//3),len(Xp)-1)

try:
    tsne=TSNE(n_components=2,perplexity=perplexity,init="pca",learning_rate="auto",max_iter=1500,random_state=RANDOM_STATE)
except TypeError:
    tsne=TSNE(n_components=2,perplexity=perplexity,init="pca",learning_rate="auto",n_iter=1500,random_state=RANDOM_STATE)

Z=tsne.fit_transform(Xp)
P=PCA(n_components=2,random_state=RANDOM_STATE).fit_transform(Xs)
meta[["tsne_1","tsne_2"]]=Z
meta[["pca_1","pca_2"]]=P
print("graph matrix:",X.shape,"perplexity:",perplexity)

In [ ]:
# t-SNE: 正常 vs 幻觉
plt.figure(figsize=(8,6))
for label,name in [(0,"Correct"),(1,"Hallucinated")]:
    m=y==label
    plt.scatter(Z[m,0],Z[m,1],s=24,alpha=.7,label=f"{name} (n={m.sum()})")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.title("Graph-level t-SNE: structural representation")
plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR/"tsne_label.png",dpi=180)
plt.show()

# 同一坐标按 response length 着色，检查长度混杂
plt.figure(figsize=(8,6))
sc=plt.scatter(Z[:,0],Z[:,1],c=meta.response_length,s=24,alpha=.75)
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.title("Same t-SNE coordinates, colored by response length")
plt.colorbar(sc,label="Response length")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"tsne_response_length.png",dpi=180)
plt.show()

# PCA 线性基线
plt.figure(figsize=(8,6))
for label,name in [(0,"Correct"),(1,"Hallucinated")]:
    m=y==label
    plt.scatter(P[m,0],P[m,1],s=24,alpha=.7,label=name)
plt.xlabel("PCA 1"); plt.ylabel("PCA 2")
plt.title("PCA baseline")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 哪些结构统计差异最大（仅解释，不参与降维）
summary=[]
for j,name in enumerate(FEATURE_NAMES):
    a,b=X[y==0,j],X[y==1,j]
    pooled=math.sqrt((a.var()+b.var())/2+1e-12)
    diff=(b.mean()-a.mean())/pooled
    summary.append([name,a.mean(),b.mean(),diff,abs(diff)])
effect=pd.DataFrame(summary,columns=["feature","correct_mean","hallucinated_mean","standardized_diff","abs_diff"])
display(effect.sort_values("abs_diff",ascending=False).head(15))

## 代表性正常图 / 幻觉图

分别选择离各自 t-SNE 类中心最近的一张图，并画 `source position → target position`。
虚线是 prompt/response 边界。坐标归一化后可比较不同长度样本。

In [ ]:
def representative(label):
    ids=np.flatnonzero(y==label)
    center=Z[ids].mean(0)
    return ids[np.argmin(((Z[ids]-center)**2).sum(1))]

def plot_pattern(g,title,max_edges=5000):
    ei,et=g["edge_index"].long(),g["edge_type"].long()
    N=g["x"].shape[0]; boundary=int(g["response_idx"])/max(1,N-1)
    s,t=ei[0].float()/max(1,N-1),ei[1].float()/max(1,N-1)
    strength=edge_strength(g)
    if len(s)>max_edges:
        keep=torch.topk(strength,max_edges).indices
        s,t,et,strength=s[keep],t[keep],et[keep],strength[keep]
    size=8+34*(strength/strength.max().clamp_min(1e-8)).numpy() if len(strength) else 12

    plt.figure(figsize=(7,6))
    for rel,name in [(0,"Prompt -> Response"),(1,"Response -> Response")]:
        m=et==rel
        plt.scatter(s[m],t[m],s=size[m] if not np.isscalar(size) else size,alpha=.55,label=name)
    plt.axvline(boundary,ls="--",alpha=.6); plt.axhline(boundary,ls="--",alpha=.6)
    plt.xlim(0,1); plt.ylim(0,1)
    plt.xlabel("Normalized source position"); plt.ylabel("Normalized target position")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.show()

ci,hi=representative(0),representative(1)
display(meta.iloc[[ci,hi]])

cg=load_graph(Path(meta.iloc[ci].graph_path))
hg=load_graph(Path(meta.iloc[hi].graph_path))
plot_pattern(cg,f"Representative correct graph: {meta.iloc[ci].sample_id}")
plot_pattern(hg,f"Representative hallucinated graph: {meta.iloc[hi].sample_id}")

In [ ]:
meta.to_csv(OUTPUT_DIR/"graph_tsne_coordinates.csv",index=False)
feature_df=pd.DataFrame(X,columns=FEATURE_NAMES)
feature_df.insert(0,"sample_id",meta.sample_id)
feature_df.insert(1,"source_id",meta.source_id)
feature_df.insert(2,"label",y)
feature_df.to_csv(OUTPUT_DIR/"graph_structure_features.csv",index=False)
print("saved to",OUTPUT_DIR.resolve())

### 如何解释

- `structure` 已分开：纯图连接模式本身含 correctness signal。
- 类别图与 length 着色图高度一致：优先做 length-controlled / source-disjoint 分析。
- `source_hhi / max_source_share` 更高：连接更集中在少数 source。
- `rr_local4 / rr_local8` 更高：更依赖局部 response history。
- 后续如果训练 GNN/GraphMAE，只需把 encoder 的 `[samples, dim]` graph embedding 替换这里的 `X`，PCA/t-SNE 部分可直接复用。

t-SNE 不需要监督标签训练，但会针对当前样本集合优化二维坐标，因此不能把图上的分离直接当成检测性能。